Here is the code where you put in H, M, G, model and measurement

In [23]:
#| default_exp prediction_module

In [24]:
#| export
import pickle
import pandas as pd 
from anthropmass.data_module import *
from anthropmass.anthro_module import *
from anthropmass.bambi_module import *

This function is normalizing the persons weight and height

In [25]:
#| export
def normalize_person(weight, height, gender):
    person=pd.DataFrame({'weightkg': [weight], 'stature': [height], 'Gender': [gender]})
    normalize(person, 'weightkg')
    normalize(person, 'stature')
    return person

This functions gets the pickled model

In [26]:
#| export
def get_pickled_model(kindofmodel:str, measurement:str):
    filepath = f'../output/anthro_models/{kindofmodel}/pickle_{measurement}_{kindofmodel}'
    with open(filepath,'rb') as file:
        model=pickle.load(file)
    return model

Predict_from_model uses the pickled model and the normalized person to predict the measurement. One of the arguments passed into this functions is "kind of model", and currently there is three different models to chose between. The three models are xgboost, bambi, and bambi with component as input. When using bambi with component as input, the component needs to passed as an argument into the functions, otherwise Army Reserve is set as default. The three different components to pass into the function are Regular Army, Army National Guard, and Army Reserve.

In [ ]:
#| export
def predict_from_model(kindofmodel:str, measurement:str, w, h, g, c=False):
    
    pickledmodel = get_pickled_model(kindofmodel, measurement)
    person = normalize_person(w, h, g)

    if kindofmodel=='xgboost':
        return pickledmodel.predict(person)
    
    elif kindofmodel=='bambi':
        model = model_bmb(measurement)
        return predict_mean_bmb(pickledmodel, model, person, measurement)
    
    elif kindofmodel=='bambi_c':
        if c!= False:
            person['Component']=c
        else:
            person['Component']='Army Reserve'
        formula='0 + C(Gender) + Component + weightkg + stature'
        model = make_model_formula(measurement, formula)
        return predict_mean_bmb(pickledmodel, model, person, measurement)
    
    else:
        return 'wrong model name'

In [63]:
#| export
def add_to_df(df, measurement, pred):
    df[measurement]=pred
    return df

I dont know if we need to make it to csv?

In [64]:
#| export
def make_csv(data, name):
    data.to_csv(f'{name}.csv', index=False)

In the following function the function above is called upon and looped through for all passed measurements.

In [66]:
#| export
def make_predictions(kindofmodel:str, measurements:list, w, h, g, c=False):
    output=pd.DataFrame()
    for m in measurements:
        pred = predict_from_model(kindofmodel, m, w, h, g, c=False)
        add_to_df(output, m, pred)
    return output

Predict for group is a function that itterates through a dataframe with multiple people. THe argument n is used when you only want to predict a part of the dataframe. 

In [83]:
def predict_for_group(kindofmodel:str, measurements:list, group:dict, n=False):
    preds_all=pd.DataFrame()
    for index, row in group.iterrows():
        print('index:', index)
        if kindofmodel == 'bambi_c':
            pred_row = make_predictions(kindofmodel, measurements, row['weightkg'], row['stature'], row['Gender'], row['Component'])
        else:
            pred_row = make_predictions(kindofmodel, measurements, row['weightkg'], row['stature'], row['Gender'])
        preds_all = pd.concat([preds_all, pred_row], ignore_index=True)
        if index == n:
            return preds_all
    return preds_all

In [86]:
test=pd.read_csv('../data/processed/ANSURIItest.csv')
test_var=test.iloc[:,1:94].drop('weightkg',axis=1).drop('stature',axis=1)
col = test_var.columns[0:1]

predict_for_group('bambi_c', col, test,3)

index: 0
index: 1
index: 2
index: 3


,abdominalextensiondepthsitting
0,248.230863
1,246.857379
2,248.286923
3,247.675919


In [68]:
make_predictions('bambi_c',['abdominalextensiondepthsitting','acromialheight'], 80, 1700, 1, 'Army National Guard')

,abdominalextensiondepthsitting,acromialheight
0,248.020508,1401.582255


In [56]:
make_predictions('bambi',['abdominalextensiondepthsitting','acromialheight'], 80, 1700, 1)

,abdominalextensiondepthsitting,acromialheight
0,250.066345,1390.5659


In [10]:
make_predictions('xgboost',['abdominalextensiondepthsitting','acromialheight','neckcircumference'], 80, 1700, 1)

,abdominalextensiondepthsitting,acromialheight,neckcircumference
0,250.98558,1392.302124,395.754181


In [ ]:
test=pd.read_csv('../data/processed/ANSURIItest.csv')


test_var=test.iloc[:,1:94].drop('weightkg',axis=1).drop('stature',axis=1)
col = test_var.columns[0:5]


In [ ]:
test_preds_all=pd.DataFrame()
for index, row in test.iterrows():
    print('index:', index)
    test_pred_xgb = make_predictions('bambi',col, row['weightkg'], row['stature'], row['Gender'])
    test_preds_all = pd.concat([test_preds_all, test_pred_xgb], ignore_index=True)
    if index == 3:
        break
test_preds_all

index: 0
index: 1
index: 2
index: 3


,abdominalextensiondepthsitting,acromialheight,acromionradialelength,anklecircumference,axillaheight
0,240.374263,1495.589415,347.716273,229.090206,1384.432253
1,209.468326,1533.136656,356.313545,223.383543,1426.359486
2,253.186293,1426.914004,332.697230,227.930137,1323.171260
3,232.316872,1477.526501,344.229518,225.951695,1375.452604


In [74]:
test_preds_all=pd.DataFrame()
for index, row in test.iterrows():
    print('index:', index)
    test_pred_xgb = make_predictions('bambi',col, row['weightkg'], row['stature'], row['Gender'], row['Component'])
    test_preds_all = pd.concat([test_preds_all, test_pred_xgb], ignore_index=True)
    if index == 10:
        break
test_preds_all

index: 0
index: 1
index: 2
index: 3
index: 4
index: 5
index: 6
index: 7
index: 8
index: 9
index: 10


,abdominalextensiondepthsitting,acromialheight,acromionradialelength,anklecircumference,axillaheight
0,240.123422,1495.851882,347.671193,229.060302,1384.515344
1,209.362462,1533.272404,356.345765,223.699219,1426.064541
2,252.787141,1427.140593,332.521799,228.237265,1323.167186
3,232.362402,1477.535942,344.304132,226.031718,1375.578251
4,236.734021,1485.403029,345.575336,227.593356,1375.170395
5,203.561336,1372.440714,319.636127,211.128522,1279.616635
6,203.691130,1290.446434,300.859172,205.362194,1200.934370
7,288.609715,1369.284992,318.644774,233.599452,1254.155331
8,211.939488,1447.518862,336.489119,218.342394,1343.333754
9,229.736294,1357.736726,315.720047,217.112539,1254.111132


In [75]:
make_csv(test_preds_all,'../output/anthro_results/predict_test_bambi_c10')

In [50]:
import nbdev; nbdev.nbdev_export()